In [2]:
EMBEDDING_MODEL_NAME = "bgebase"

TICKERS = [
    "AUR",
    "TSLA",
    "MBLY",
    "GOOGL",
    "GM",
    "F",
    "NVDA",
    "QCOM",
    "APTV",
    "OUST",
    "RIVN",
]

In [3]:
from pathlib import Path
import json
import sys
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings.embed_chunks import MODEL_CONFIGS

EMBEDDING_CONFIG = MODEL_CONFIGS[EMBEDDING_MODEL_NAME]
QUERY_PREFIX = EMBEDDING_CONFIG["query_prefix"]

EMBEDDINGS_DIR = PROJECT_ROOT / "data/embeddings"
CHUNKS_DIR = PROJECT_ROOT / "data/chunks"

FILINGS = {
    "AUR": "2025-10-K",
    "TSLA": "2025-10-K",
    "MBLY": "2025-10-K",
    "GOOGL": "2025-10-K",
    "GM": "2025-10-K",
    "F": "2025-10-K",
    "NVDA": "2026-10-K",
    "QCOM": "2025-10-K",
    "APTV": "2025-10-K",
    "OUST": "2025-10-K",
    "RIVN": "2025-10-K",
}

company_embeddings = []
all_chunks = []

for ticker, filing_name in FILINGS.items():
    embedding_paths = list(
        (EMBEDDINGS_DIR / ticker).glob(
            f"{filing_name}.{EMBEDDING_MODEL_NAME}*.npz"
        )
    )
    if len(embedding_paths) != 1:
        raise ValueError(
            f"Expected one {EMBEDDING_MODEL_NAME} embedding file for "
            f"{ticker}/{filing_name}, found: {embedding_paths}"
        )
    embedding_path = embedding_paths[0]

    data = np.load(embedding_path)
    embeddings = data["embeddings"]

    chunk_path = (
        CHUNKS_DIR / ticker / f"{filing_name}.chunks.jsonl"
    )

    with open(chunk_path, "r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f if line.strip()]

    assert len(embeddings) == len(chunks), (
        f"{ticker}: {len(embeddings)} embeddings != {len(chunks)} chunks"
    )

    company_embeddings.append(embeddings)
    all_chunks.extend(chunks)

all_embeddings = np.vstack(company_embeddings)

assert len(all_embeddings) == len(all_chunks)

print("Total chunks:", len(all_chunks))
print("Embedding matrix:", all_embeddings.shape)

/data/Programiranje/Sixsentix/edgar_insight_rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 4115
Embedding matrix: (4115, 768)


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_CONFIG["repository"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4809.76it/s]


In [4]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4567.92it/s]


In [73]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7432.10it/s]


In [5]:
import numpy as np

from src.embeddings.embed_chunks import table_embedding_text


# ============================================================
# PRECOMPUTE NORMALIZED CORPUS EMBEDDINGS ONCE
# ============================================================

embedding_norms = np.linalg.norm(
    all_embeddings,
    axis=1,
    keepdims=True,
)

normalized_embeddings = (
    all_embeddings / np.clip(embedding_norms, 1e-12, None)
)


# ============================================================
# RERANKER INPUT
# ============================================================

def get_rerank_text(chunk):
    """
    Narrative:
        explicit company metadata + original chunk text

    Table:
        explicit company metadata
        + compact semantic table representation
        + full original Markdown table

    The compact representation comes first so the important
    semantic information is retained even if the cross-encoder
    truncates a very large table.
    """

    prefix = (
        f"Company: {chunk.get('company', '')}\n"
        f"Ticker: {chunk.get('ticker', '')}\n"
        f"Section: {chunk.get('section', '')}\n"
        f"Content type: {chunk.get('content_type', '')}\n"
    )

    if chunk.get("content_type") == "table":
        compact_table = table_embedding_text(chunk)

        return (
            prefix
            + "\n"
            + compact_table
            + "\n\nFull table:\n"
            + chunk.get("text", "")
        )

    return prefix + "\n" + chunk.get("text", "")


# ============================================================
# RETRIEVAL + RERANKING
# ============================================================

def retrieve_candidates(
    query,
    model,
    reranker,
    normalized_embeddings,
    all_chunks,
    top_k=30,
    table_top_k=10,
    final_top_k=10,
    reranker_batch_size=32,
):
    # --------------------------------------------------------
    # 1. Embed query
    # --------------------------------------------------------

    query_text = QUERY_PREFIX + query

    query_embedding = model.encode(
        query_text,
        normalize_embeddings=True,
    )

    # --------------------------------------------------------
    # 2. Global dense retrieval
    # --------------------------------------------------------

    dense_scores = normalized_embeddings @ query_embedding

    global_indices = np.argsort(
        dense_scores
    )[-top_k:][::-1]

    global_rank = {
        int(idx): rank
        for rank, idx in enumerate(global_indices, start=1)
    }

    # --------------------------------------------------------
    # 3. Table-only dense retrieval
    # --------------------------------------------------------

    table_indices = np.array([
        i
        for i, chunk in enumerate(all_chunks)
        if chunk.get("content_type") == "table"
    ])

    table_top_indices = []
    table_rank = {}

    if len(table_indices) > 0:
        table_embeddings = normalized_embeddings[table_indices]

        table_scores = table_embeddings @ query_embedding

        local_indices = np.argsort(
            table_scores
        )[-table_top_k:][::-1]

        table_top_indices = [
            int(table_indices[i])
            for i in local_indices
        ]

        table_rank = {
            idx: rank
            for rank, idx in enumerate(
                table_top_indices,
                start=1,
            )
        }

    # --------------------------------------------------------
    # 4. Merge global + table candidates and deduplicate
    # --------------------------------------------------------

    candidate_indices = []
    seen = set()

    for idx in list(global_indices) + table_top_indices:
        idx = int(idx)

        if idx not in seen:
            candidate_indices.append(idx)
            seen.add(idx)

    # --------------------------------------------------------
    # 5. Build candidate objects
    # --------------------------------------------------------

    results = []

    for idx in candidate_indices:
        chunk = all_chunks[idx]

        results.append({
            "index": idx,
            "chunk": chunk,
            "dense_score": float(dense_scores[idx]),
            "global_dense_rank": global_rank.get(idx),
            "table_dense_rank": table_rank.get(idx),
            "from_global": idx in global_rank,
            "from_table_search": idx in table_rank,
        })

    # --------------------------------------------------------
    # 6. Cross-encoder reranking
    # --------------------------------------------------------

    pairs = [
        (
            query,
            get_rerank_text(result["chunk"]),
        )
        for result in results
    ]

    reranker_scores = reranker.predict(
        pairs,
        batch_size=reranker_batch_size,
        show_progress_bar=False,
    )

    for result, score in zip(results, reranker_scores):
        result["reranker_score"] = float(score)

    # --------------------------------------------------------
    # 7. Final reranked ordering
    # --------------------------------------------------------

    results.sort(
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    for rank, result in enumerate(results, start=1):
        result["rerank"] = rank

    return results[:final_top_k]


# ============================================================
# PRINT RESULTS
# ============================================================

def print_results(results):
    for result in results:
        chunk = result["chunk"]

        print("=" * 120)

        print(
            f"{result['rerank']:2}. "
            f"{chunk.get('ticker')} | "
            f"type={chunk.get('content_type')} | "
            f"dense={result['dense_score']:.4f} | "
            f"rerank={result['reranker_score']:.4f}"
        )

        print(
            f"global_rank={result['global_dense_rank']} | "
            f"table_rank={result['table_dense_rank']} | "
            f"global={result['from_global']} | "
            f"table_search={result['from_table_search']}"
        )

        print(
            f"Section: {chunk.get('section')}"
        )

        print("-" * 120)

        # Print complete tables so values can be inspected.
        if chunk.get("content_type") == "table":
            print(chunk.get("text", ""))

        # Narrative output can stay truncated for readability.
        else:
            print(chunk.get("text", "")[:2000])

        print()

In [6]:
def load_test_queries(path):
    test_cases = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            try:
                test_cases.append(json.loads(line))

            except json.JSONDecodeError as e:
                print(f"Invalid JSON on line {line_number}")
                print(f"Error: {e}")
                print("\nLine:")
                print(line)
                print("\nNear error:")
                start = max(0, e.pos - 50)
                end = min(len(line), e.pos + 50)
                print(line[start:end])
                raise

    return test_cases

In [7]:
TEST_QUERIES_PATH = Path("../data/evaluation/test_queries.jsonl")
load_test_queries(TEST_QUERIES_PATH)

[{'company': 'Aurora Innovation, Inc.',
  'ticker': 'AUR',
  'question': "What is Aurora's mission for delivering self-driving technology?",
  'question_type': 'single_narrative',
  'expected_chunk_ids': ['AUR-2025-CHUNK-000002']},
 {'company': 'Aurora Innovation, Inc.',
  'ticker': 'AUR',
  'question': 'Who founded Aurora in 2017?',
  'question_type': 'single_narrative',
  'expected_chunk_ids': ['AUR-2025-CHUNK-000002']},
 {'company': 'Aurora Innovation, Inc.',
  'ticker': 'AUR',
  'question': 'What vehicle types is the Aurora Driver designed to work across?',
  'question_type': 'single_narrative',
  'expected_chunk_ids': ['AUR-2025-CHUNK-000002']},
 {'company': 'Aurora Innovation, Inc.',
  'ticker': 'AUR',
  'question': 'Which three target markets does Aurora identify for the Aurora Driver?',
  'question_type': 'single_narrative',
  'expected_chunk_ids': ['AUR-2025-CHUNK-000002']},
 {'company': 'Aurora Innovation, Inc.',
  'ticker': 'AUR',
  'question': 'Why did Aurora choose truckin

In [8]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

TEST_QUERIES_PATH = Path("../data/evaluation/test_queries.jsonl")

K_VALUES = [1, 3, 5, 10, 30]
MAX_K = max(K_VALUES)


# ============================================================
# PURE DENSE RETRIEVAL
# ============================================================

def dense_retrieve(query, top_k=30):
    start = time.perf_counter()

    query_embedding = model.encode(
        QUERY_PREFIX + query,
        normalize_embeddings=True,
    )

    scores = normalized_embeddings @ query_embedding

    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = [
        {
            "rank": rank,
            "chunk_id": all_chunks[idx]["chunk_id"],
            "ticker": all_chunks[idx].get("ticker"),
            "content_type": all_chunks[idx].get("content_type"),
            "score": float(scores[idx]),
            "index": int(idx),
        }
        for rank, idx in enumerate(top_indices, start=1)
    ]

    latency = time.perf_counter() - start

    return results, latency


# ============================================================
# METRICS FOR ONE QUERY
# ============================================================

def evaluate_query(test_case, retrieved):
    expected_ids = set(test_case["expected_chunk_ids"])

    retrieved_ids = [
        result["chunk_id"]
        for result in retrieved
    ]

    # Rank of each expected chunk, if retrieved
    expected_ranks = {
        chunk_id: (
            retrieved_ids.index(chunk_id) + 1
            if chunk_id in retrieved_ids
            else None
        )
        for chunk_id in expected_ids
    }

    row = {
        "company": test_case["company"],
        "ticker": test_case["ticker"],
        "question_type": test_case["question_type"],
        "question": test_case["question"],
        "expected_chunk_ids": list(expected_ids),
        "expected_chunk_count": len(expected_ids),
        "expected_ranks": expected_ranks,
    }

    for k in K_VALUES:
        top_k_ids = set(retrieved_ids[:k])

        relevant_found = expected_ids & top_k_ids
        relevant_count = len(relevant_found)

        # Fraction of required evidence retrieved
        recall = relevant_count / len(expected_ids)

        # At least one required chunk found
        hit = int(relevant_count > 0)

        # ALL required chunks found
        complete = int(relevant_count == len(expected_ids))

        # Reciprocal rank of first relevant result
        relevant_ranks = [
            rank
            for chunk_id, rank in expected_ranks.items()
            if rank is not None and rank <= k
        ]

        reciprocal_rank = (
            1.0 / min(relevant_ranks)
            if relevant_ranks
            else 0.0
        )

        row[f"recall@{k}"] = recall
        row[f"hit@{k}"] = hit
        row[f"complete@{k}"] = complete
        row[f"mrr@{k}"] = reciprocal_rank

    return row


# ============================================================
# RUN FULL EVALUATION
# ============================================================

def evaluate_dense_retrieval():
    test_cases = load_test_queries(TEST_QUERIES_PATH)

    # Validate ground-truth IDs before evaluating
    corpus_chunk_ids = {
        chunk["chunk_id"]
        for chunk in all_chunks
    }

    missing_gold_ids = set()

    for case in test_cases:
        for chunk_id in case["expected_chunk_ids"]:
            if chunk_id not in corpus_chunk_ids:
                missing_gold_ids.add(chunk_id)

    if missing_gold_ids:
        raise ValueError(
            "Ground-truth chunk IDs missing from current corpus:\n"
            + "\n".join(sorted(missing_gold_ids))
        )

    rows = []

    total_start = time.perf_counter()

    for i, case in enumerate(test_cases, start=1):
        retrieved, latency = dense_retrieve(
            case["question"],
            top_k=MAX_K,
        )

        row = evaluate_query(
            case,
            retrieved,
        )

        row["latency_seconds"] = latency

        # Useful for failure inspection later
        row["retrieved_chunk_ids"] = [
            result["chunk_id"]
            for result in retrieved
        ]

        rows.append(row)

        if i % 25 == 0 or i == len(test_cases):
            print(
                f"Evaluated {i}/{len(test_cases)} queries"
            )

    total_time = time.perf_counter() - total_start

    df = pd.DataFrame(rows)

    return df, total_time


# ============================================================
# SUMMARY
# ============================================================

def metric_columns():
    columns = []

    for k in K_VALUES:
        columns.extend([
            f"recall@{k}",
            f"hit@{k}",
            f"complete@{k}",
            f"mrr@{k}",
        ])

    return columns


def print_evaluation_summary(df, total_time):
    metrics = metric_columns()

    print("\n" + "=" * 90)
    print("OVERALL DENSE RETRIEVAL")
    print("=" * 90)

    overall = df[metrics].mean()

    for k in K_VALUES:
        print(
            f"@{k:<2} | "
            f"Recall={overall[f'recall@{k}']:.4f} | "
            f"Hit={overall[f'hit@{k}']:.4f} | "
            f"Complete={overall[f'complete@{k}']:.4f} | "
            f"MRR={overall[f'mrr@{k}']:.4f}"
        )

    print("\nLatency")
    print(f"Queries: {len(df)}")
    print(f"Total:   {total_time:.3f}s")
    print(f"Mean:    {df['latency_seconds'].mean():.4f}s")
    print(f"Median:  {df['latency_seconds'].median():.4f}s")
    print(f"P95:     {df['latency_seconds'].quantile(0.95):.4f}s")

    # --------------------------------------------------------
    # By question type
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("BY QUESTION TYPE")
    print("=" * 90)

    type_summary = (
        df.groupby("question_type")[metrics]
        .mean()
        .round(4)
    )

    print("Question type | " + " | ".join(metrics))
    print("=" * 90)
    for question_type, row in type_summary.iterrows():
        values = " | ".join(
            f"{row[metric]:.4f}"
            for metric in metrics
        )
        print(f"{question_type} | {values}")

    # --------------------------------------------------------
    # By company
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("BY COMPANY")
    print("=" * 90)

    company_summary = (
        df.groupby(["ticker", "company"])[metrics]
        .mean()
        .round(4)
    )

    print("Ticker | Company | " + " | ".join(metrics))
    print("=" * 90)
    for (ticker, company), row in company_summary.iterrows():
        values = " | ".join(
            f"{row[metric]:.4f}"
            for metric in metrics
        )
        print(f"{ticker} | {company} | {values}")

In [1]:
# ============================================================
# RUN
# ============================================================
evaluation_df, total_time = evaluate_dense_retrieval()

print_evaluation_summary(
    evaluation_df,
    total_time,
)


# ============================================================
# SAVE DETAILED RESULTS
# ============================================================

from datetime import datetime
import re
import subprocess

embedding_model = model[0].auto_model.config._name_or_path
model_name = re.sub(
    r"[^A-Za-z0-9._-]+",
    "-",
    embedding_model.rsplit("/", maxsplit=1)[-1],
).strip(".-")
OUTPUT_DIR = Path("../data/evaluation") / model_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using model: {model_name.upper()}")

existing_versions = [
    int(match.group(1))
    for path in OUTPUT_DIR.glob("dense_retrieval_evaluation_v*.jsonl")
    if (match := re.fullmatch(
        r"dense_retrieval_evaluation_v(\d+)\.jsonl",
        path.name,
    ))
]
run_number = max(existing_versions, default=0) + 1
OUTPUT_PATH = OUTPUT_DIR / f"dense_retrieval_evaluation_v{run_number}.jsonl"

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        text=True,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None

run_metadata = {
    "run_number": run_number,
    "timestamp": datetime.now().astimezone().isoformat(),
    "embedding_model": embedding_model,
    "reranker_model": None,
    "retrieval_method": dense_retrieve.__name__,
    "retrieval_parameters": {
        "top_k": MAX_K,
        "query_prefix": QUERY_PREFIX,
    },
    "evaluation_dataset": str(TEST_QUERIES_PATH),
    "git_commit": git_commit,
    "device": str(model.device),
}

with open(OUTPUT_PATH, "x", encoding="utf-8") as f:
    for record in evaluation_df.to_dict(orient="records"):
        record = {**record, **run_metadata}
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"\nSaved detailed results to: {OUTPUT_PATH}")

NameError: name 'evaluate_dense_retrieval' is not defined

In [10]:
import json


EVALUATION_PATH = "../data/evaluation/bge-base-en-v1.5/dense_retrieval_evaluation_v3.jsonl"


def print_company_retrieval_results(company_name):
    with open(EVALUATION_PATH, "r", encoding="utf-8") as f:
        records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    company_records = [
        record
        for record in records
        if record["company"].lower() == company_name.lower()
    ]

    if not company_records:
        print(f"No results found for company: {company_name}")
        return

    for i, record in enumerate(company_records, start=1):
        question_type = record["question_type"]

        if question_type == "single_narrative":
            display_type = "narrative"
        elif question_type == "single_table":
            display_type = "table"
        else:
            display_type = "multi-chunk"

        expected_ids = record["expected_chunk_ids"]
        expected_ranks = record["expected_ranks"]

        retrieved = [
            (chunk_id, expected_ranks.get(chunk_id))
            for chunk_id in expected_ids
            if expected_ranks.get(chunk_id) is not None
        ]

        ranks = [
            rank
            for _, rank in retrieved
        ]

        print(f"{i}. Query: {record['question']}")
        print(f"   Type: {display_type} ({question_type})")
        print(
            f"   Retrieved required chunks: "
            f"{len(retrieved)}/{len(expected_ids)}"
        )
        print(f"   Ranks: {ranks if ranks else 'None'}")

        # Helpful for multi-chunk failures
        if len(retrieved) < len(expected_ids):
            missing = [
                chunk_id
                for chunk_id in expected_ids
                if expected_ranks.get(chunk_id) is None
            ]
            print(f"   Missing: {missing}")

        print()

In [11]:
print_company_retrieval_results("Ford Motor Company")

1. Query: When was Ford Motor Company incorporated in Delaware, and where is the company headquartered?
   Type: narrative (single_narrative)
   Retrieved required chunks: 1/1
   Ranks: [1]

2. Query: How does Ford distribute its vehicles, parts, and accessories to retail customers?
   Type: narrative (single_narrative)
   Retrieved required chunks: 1/1
   Ranks: [2]

3. Query: Which base metals, battery materials, and rare earth minerals does Ford identify among its raw-material purchases?
   Type: narrative (single_narrative)
   Retrieved required chunks: 1/1
   Ranks: [1]

4. Query: What types of vehicle-related financing and leasing activities make up Ford Credit's business?
   Type: narrative (single_narrative)
   Retrieved required chunks: 1/1
   Ranks: [1]

5. Query: What categories of governmental standards and regulations apply to Ford's vehicles and manufacturing facilities?
   Type: narrative (single_narrative)
   Retrieved required chunks: 1/1
   Ranks: [1]

6. Query: How c

In [12]:
query = "Who seems to be targeting robotaxi services, and who is more focused on freight?"
TOP_K = 10
print("Query: ", query)
print(f"Retrieving top {TOP_K} chunks..")
retrieval_query = QUERY_PREFIX + query

query_embedding = model.encode(
    retrieval_query,
    normalize_embeddings=True,
)

scores = normalized_embeddings @ query_embedding

top_indices = np.argsort(scores)[-TOP_K:][::-1]

for rank, idx in enumerate(top_indices, start=1):
    chunk = all_chunks[idx]

    print("=" * 120)

    print(
        f"{rank:2}. "
        f"{chunk.get('ticker')} | "
        f"type={chunk.get('content_type')} | "
        f"dense={scores[idx]:.4f}"
    )

    print(f"Chunk ID: {chunk.get('chunk_id')}")
    print(f"Section:  {chunk.get('section')}")
    print("-" * 120)

    if chunk.get("content_type") == "table":
        print(chunk.get("text", ""))
    else:
        print(chunk.get("text", "")[:2000])

    print()

Query:  Who seems to be targeting robotaxi services, and who is more focused on freight?
Retrieving top 30 chunks..
 1. MBLY | type=narrative | dense=0.6152
Chunk ID: MBLY-2025-CHUNK-000105
Section:  Item 1A — Risk Factors
------------------------------------------------------------------------------------------------------------------------
Item 1A — Risk Factors
Risks Related to Our Business

Our reliance on third-party partners for robotaxi deployments exposes us to risks associated with complex and evolving commercial relationships, including disagreements regarding deployment schedules, operational responsibilities, economics, branding, data access and use, liability allocation, and the allocation of costs associated with vehicle hardware, software updates, maintenance and fleet operations. Our partners may also face their own operational, financial or strategic challenges, may decide to delay, reduce or discontinue their autonomous mobility programs, may pursue competing technolo